In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import KBinsDiscretizer

In [2]:
df = pd.read_csv("data/final_results.csv")

In [3]:
df.head()

,gemeente,year,total_accidents,total_cycling_accidents,car_conflict,severity,slick,light,intersection,school_peak,...,historical_town_avg,school_zone_exposure,risk_per_1k_residents,weather_severity_index,flow_density,smoothed_target,predicted_num_accidents,pred_per_1k_residents,risk_rank,most_frequent_type
0,Gent,2025,0,0,0,0,0,0,0,0,...,442.80,6.401417,4.694543,0.604669,0.016337,0,437,1.602746,1.0,dark
1,Brugge,2025,0,0,0,0,0,0,0,0,...,164.40,2.874811,4.298197,0.650925,0.033503,0,223,1.853961,2.0,intersection
2,Leuven,2025,0,0,0,0,0,0,0,0,...,140.00,0.817294,3.488838,0.651850,0.114405,0,213,2.030389,3.0,slick
3,Kortrijk,2025,0,0,0,0,0,0,0,0,...,98.60,1.238004,4.159651,0.886546,0.091269,0,136,1.683668,4.0,car_conflict
4,Mechelen,2025,0,0,0,0,0,0,0,0,...,79.25,4.801774,3.045006,0.633077,0.025570,0,133,1.483464,5.0,slick


In [4]:
df_tiers = df[["gemeente", "predicted_num_accidents", "total_pop"]].copy()

In [5]:
df_tiers["people_that_cycle"] = (1 - 0.29) * df_tiers["total_pop"]
df_tiers["accidents_per_100_cyclists"] = df_tiers["predicted_num_accidents"] / (
    df_tiers["people_that_cycle"] / 100
)

In [6]:
df_tiers["accidents_per_100_cyclists"] = (
    df_tiers["accidents_per_100_cyclists"].fillna(0).replace([np.inf, -np.inf], 0)
)

In [7]:
X = df_tiers[["accidents_per_100_cyclists"]].values

In [8]:
discretizer = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="kmeans", random_state=7)

In [9]:
df_tiers["bin_encoded"] = discretizer.fit_transform(X).astype(int)

In [10]:
color_mapping = {
    0: "Green",
    1: "Yellow",
    2: "Red"
}

In [11]:
df_tiers["risk_tier"] = df_tiers["bin_encoded"].map(color_mapping)

In [12]:
bin_edges = discretizer.bin_edges_[0]
print(f"Green Tier Range:  {bin_edges[0]:.4f} to {bin_edges[1]:.4f}")
print(f"Yellow Tier Range: {bin_edges[1]:.4f} to {bin_edges[2]:.4f}")
print(f"Red Tier Range:    {bin_edges[2]:.4f} to {bin_edges[3]:.4f}\n")

Green Tier Range:  0.0153 to 0.1252
Yellow Tier Range: 0.1252 to 0.2031
Red Tier Range:    0.2031 to 0.2946



In [13]:
df_tiers

,gemeente,predicted_num_accidents,total_pop,people_that_cycle,accidents_per_100_cyclists,bin_encoded,risk_tier
0,Gent,437,272657.0,193586.47,0.225739,2,Red
1,Brugge,223,120283.0,85400.93,0.261121,2,Red
2,Leuven,213,104906.0,74483.26,0.285970,2,Red
3,Kortrijk,136,80776.0,57350.96,0.237136,2,Red
4,Mechelen,133,89655.0,63655.05,0.208939,2,Red
...,...,...,...,...,...,...,...
63,Wemmel,5,18292.0,12987.32,0.038499,0,Green
64,Sint-Pieters-Leeuw,4,36732.0,26079.72,0.015338,0,Green
65,Heers,3,7574.0,5377.54,0.055788,0,Green
66,Kraainem,3,13940.0,9897.40,0.030311,0,Green


In [14]:
df_tiers.to_csv("data/tiers.csv", index=False)

In [15]:
df_tiers[df_tiers["gemeente"] == "Heist-op-den-Berg"]

,gemeente,predicted_num_accidents,total_pop,people_that_cycle,accidents_per_100_cyclists,bin_encoded,risk_tier
22,Heist-op-den-Berg,41,44723.0,31753.33,0.12912,1,Yellow
